# 2 · Conjuring geometry & taming the mesh ⚔️🛡️

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [⌛ ~1 min](/lite/notebooks/index.html?path=02-geometry.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/02-geometry.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** to
switch story / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — to the forge
:class: storytelling

*Before any adventurer rides out, they visit the **forge**. The Beast you have met; now
you must arm yourself. You will **hammer out a sword and a shield** — not from steel, but
from pure geometry — and learn, in the doing, how to **sketch shapes** and **tame the mesh**
that NGSolve lays over them.*
:::

Unit 1 began in **3D** (the Beast). Here we work the other dimensions: **2D** sketches
(a sword and a shield), the **knobs** that control a mesh, **1D** meshes, **mesh-topology
queries**, and **importing a real external model**. The 2D sword & shield return in **3D**
in the first supplement.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
# --- Bring the website's UI into this live notebook: the ⚙ View-options panel,
# the foldable story/how-to/quiz/further-reading categories and the gimmicks
# (rolling logo + winking head). Loads static/custom.css + view-options.js via
# notebooks/data/ngsum_ui.py. A no-op on the static-site build. --------------
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    sys.path.insert(0, os.path.join(os.getcwd(), "data"))
    try:
        import ngsum_ui; ngsum_ui.enable()
    except Exception:
        pass

In [ ]:
from netgen.occ import WorkPlane, OCCGeometry, Axis, Pnt, X, Y, Z, Glue
from ngsolve import Mesh, H1, GridFunction, x, y
from ngsolve.meshes import Make1DMesh
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt

## 1. Forge a sword — a 2D sketch

The 2D workflow is **sketch → face**. A `WorkPlane` is a pen on a sheet of paper:
`MoveTo` puts it down, each `LineTo` draws a straight segment, `Close` joins back to the
start, and `.Face()` fills the closed outline. We trace the **silhouette of a sword** in
one loop — blade, crossguard, grip, pommel.

:::{important}
**Orientation matters.** A face's outline must run **counter-clockwise** (so it encloses a
*positive* area). Trace it clockwise and the area comes out negative — the mesher then
silently fails to triangulate it. We walk **tip → left → pommel → right → back**.
:::

In [ ]:
def sword_2d():
    """Silhouette of a sword as one counter-clockwise outline."""
    return (WorkPlane()
            .MoveTo(0, 10)                                            # the tip
            .LineTo(-0.5, 2.2).LineTo(-2.2, 2.2).LineTo(-2.2, 1.5)    # left blade edge → guard
            .LineTo(-0.35, 1.5).LineTo(-0.35, -1.6)                   # into the grip, down
            .LineTo(-0.95, -2.2).LineTo(0, -2.95).LineTo(0.95, -2.2)  # the diamond pommel
            .LineTo(0.35, -1.6).LineTo(0.35, 1.5)                     # back up the grip
            .LineTo(2.2, 1.5).LineTo(2.2, 2.2).LineTo(0.5, 2.2)       # guard → right blade edge
            .Close().Face())

sword = sword_2d()
print(f"the sword: area {sword.mass:.1f}  (positive → correctly oriented)")
mesh_sw = Mesh(OCCGeometry(sword, dim=2).GenerateMesh(maxh=0.4))
print(f"meshed: {mesh_sw.ne} triangles")
Draw(mesh_sw)

## 2. A shield — curved edges and the meshing knobs

A straight-line sketch is easy; a **curved** boundary is where meshing gets interesting.
`Spline` draws a smooth curve through points — we use it for the rounded bottom of a
**heater shield**. Two knobs control the mesh:

* **`GenerateMesh(maxh=…)`** — the global maximum element size. Smaller ⇒ more, smaller
  triangles ⇒ more accurate but more expensive.
* **`mesh.Curve(order)`** — straight-sided elements approximate a curve by a crude
  **polygon**; `Curve` *bends* the element edges to follow the true boundary.

In [ ]:
def shield_2d(W=3.0, H=3.5):
    """A heater shield: flat top, sides splining down to a point."""
    return (WorkPlane()
            .MoveTo(-W, H).LineTo(-W, 0.3)                            # top-left, down left side
            .Spline([(-W*0.6, -2.0), (0, -H)])                       # curve to the bottom point
            .Spline([(W*0.6, -2.0), (W, 0.3)])                       # curve up the right side
            .LineTo(W, H).Close().Face())                            # up right, close the top

shield = shield_2d()
for h in (0.8, 0.35):
    m = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=h))
    print(f"maxh={h}:  {m.ne:4d} triangles")

mesh_sh = Mesh(OCCGeometry(shield, dim=2).GenerateMesh(maxh=0.35))
mesh_sh.Curve(3)                                                     # bend elements onto the curve
Draw(mesh_sh)

**Local refinement.** You rarely want the *whole* mesh fine. Setting **`.maxh` on a single
sub-shape** refines only there — here the **sharp tip** of the blade, where the geometry is
thin and the solution will vary fastest, while the broad grip stays coarse.

In [ ]:
sword_ref = sword_2d()
sword_ref.edges.Max(Y).maxh = 0.12                                  # fine only near the tip edges
mesh_ref = Mesh(OCCGeometry(sword_ref, dim=2).GenerateMesh(maxh=0.6))
print(f"locally refined sword: {mesh_ref.ne} triangles (fine tip, coarse grip)")
Draw(mesh_ref)

## 3. Down a dimension — 1D meshes

A mesh can be **one-dimensional** too — a chain of intervals on $[0,1]$. `Make1DMesh(n)`
spaces $n$ of them **uniformly**; a `mapping` grades them, packing points where you need
resolution (say near a boundary layer).

In [ ]:
def nodes(m): return sorted(p[0] for p in m.ngmesh.Points())
uniform = Make1DMesh(12)
graded  = Make1DMesh(12, mapping=lambda t: t**1.7)                  # clustered near x=0
fig, ax = plt.subplots(figsize=(7, 1.4))
ax.plot(nodes(uniform), [1]*13, "o-", label="uniform")
ax.plot(nodes(graded),  [0]*13, "o-", label="graded $t^{1.7}$")
ax.set_yticks([0, 1]); ax.set_yticklabels(["graded", "uniform"]); ax.set_xlabel("x")
ax.legend(loc="center right"); ax.set_ylim(-0.5, 1.5); fig.tight_layout()

## 4. Knowing your mesh — topology & a value at a point

A mesh is not just a picture: you can **query its topology** and **evaluate functions on
it**. `mesh.nv / mesh.ne` count vertices and elements; `mesh(x, y)` locates the element
containing a point so a `CoefficientFunction` can be evaluated there (the bridge to unit 3).

In [ ]:
print(f"the sword mesh has {mesh_sw.nv} vertices and {mesh_sw.ne} triangles")
gf = GridFunction(H1(mesh_sw, order=2))                              # a function living on the mesh
gf.Set(x*x + y*y)                                                    # = squared distance from the hilt
for px, py in [(0, 8), (0, 0), (1.5, 1.7)]:                          # tip, centre, guard
    print(f"  at ({px:>4},{py:>4}):  x²+y² = {gf(mesh_sw(px, py)):6.2f}")
Draw(gf, mesh_sw, "x²+y²")

---
## Supplementary A — the sword & shield in 3D

*(Supplementary.)* A 2D face becomes a **3D solid** by **extrusion**: `face.Extrude(d)`
pulls it a depth `d` along its normal. We give the sword and shield a thickness, move them
into a little **scene** (the shield up front, the sword angled behind it), and mesh the
pair together.

In [ ]:
sword3d  = sword_2d().Extrude(0.5)
shield3d = shield_2d().Extrude(1.0).Move((0, 0, -3))                 # shield set behind the blade
sword3d  = sword3d.Rotate(Axis(Pnt(0, 0, 0), Z), 25).Move((1.5, -1, -6))
scene = Glue([sword3d, shield3d])
mesh3d = Mesh(OCCGeometry(scene).GenerateMesh(maxh=1.2)); mesh3d.Curve(2)
print(f"the 3D loadout: {mesh3d.ne} tetrahedra")
Draw(mesh3d)

---
## Supplementary B — importing a real external model

*(Supplementary.)* Geometry need not be sketched by hand — NGSolve reads common CAD/mesh
formats. Here we **import an `.stl` model** (a surface triangulation — a blocky *Minecraft
sword*) and let Netgen build a **volume mesh** from it. The same `OCCGeometry` route reads
**STEP/IGES/BREP** CAD files, and imported parts can be **combined** with sketched ones via
the boolean operators (`+`, `-`, `*`, `Glue`) you have already met.

In [ ]:
from netgen.stl import STLGeometry
imported = STLGeometry("data/minecraft-sword.stl")
mesh_mc = Mesh(imported.GenerateMesh(maxh=12))                       # coarse: the model is detailed
print(f"imported Minecraft sword: {mesh_mc.ne} tetrahedra, {mesh_mc.nv} vertices")
Draw(mesh_mc)

:::{dropdown} 📚 Further reading
:class: further-reading

- **Netgen/OCC geometry** — the i-tutorials on
  [OpenCASCADE geometry](https://docu.ngsolve.org/latest/i-tutorials/unit-4.4-occ/occ.html)
  and [workplanes](https://docu.ngsolve.org/latest/i-tutorials/unit-4.4-occ/workplane.html).
- **Meshing options** — `maxh`, local `.maxh`, `Curve`, and named regions in the
  [NGSolve docs](https://docu.ngsolve.org/latest/).
- **A guided OCC walkthrough** — the ngs24 tutorial
  [OpenCASCADE Technology geometry](https://docu.ngsolve.org/ngs24/tutorials/03_occ.html).
:::

:::{dropdown} 🧠 Quiz — why did the first sketch refuse to mesh?
:class: quiz
Because its outline ran **clockwise**: the enclosing face then has a **negative area**, and
Netgen cannot triangulate an inside-out face (it fails quietly). Reverse the traversal so the
boundary runs **counter-clockwise** — positive area — and it meshes. The same rule governs
holes: an outer boundary CCW, an inner hole boundary CW.
:::

**Armed and ready.** You can sketch in 2D, refine where it matters, drop to 1D, query a mesh
and import a model. Next we meet the one object NGSolve evaluates *everywhere* — the
**CoefficientFunction**.

In [ ]:
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    _nb, _title = "03-coefficientfunctions", "3 · What is a CoefficientFunction? 🔨"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))